<a href="https://colab.research.google.com/github/AlexandreLouzada/exercicios-analise-dados/blob/master/sqlalchemy_rh_GABARITO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏢 Desenvolvendo um Sistema de RH com SQLAlchemy — GABARITO

**Roteiro:** SQL puro com segurança → SQLAlchemy Core → ORM com relacionamentos e sessões.

**Colab:** `Arquivo > Fazer upload do notebook`, executar com `Shift + Enter`. `sqlalchemy` + `pandas` já vêm no Colab.

---

In [ ]:
import sqlalchemy
print("SQLAlchemy:", sqlalchemy.__version__)

from sqlalchemy import create_engine, text, Table, MetaData, Column, Integer, String, Float, ForeignKey, insert, update, select, func, Numeric
from sqlalchemy.orm import declarative_base, relationship, sessionmaker, mapped_column, Session
import pandas as pd

## 🟢 Nível 1 — Configuração e SQL Puro com Segurança (Básico)

In [ ]:
# Passo 1: conexão com banco SQLite local
engine = create_engine('sqlite:///sistema_rh.db')

# Passo 2: CREATE TABLE em SQL puro dentro de transação automática
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS funcionarios (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            cargo TEXT NOT NULL,
            salario FLOAT NOT NULL
        )
    """))
print("Tabela 'funcionarios' criada (SQL puro).")

In [ ]:
# Passo 3: INSERT seguro usando placeholders (:nome, :cargo, :salario)
novo_funcionario = {"nome": "Ana Souza", "cargo": "Desenvolvedor Júnior", "salario": 3500.00}

with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO funcionarios (nome, cargo, salario) VALUES (:nome, :cargo, :salario)"),
        novo_funcionario
    )
print("Inserção segura realizada.")

In [ ]:
# Passo 4: validar lendo com pd.read_sql_query -> já retorna DataFrame
df_funcionarios = pd.read_sql_query("SELECT * FROM funcionarios", engine)
df_funcionarios

> **💬 Pergunta reflexiva — Segurança:**
> **Nunca concatene strings diretamente no SQL** porque um usuário mal-intencionado pode "quebrar" a string e injetar comandos (ex: `' OR 1=1; DROP TABLE funcionarios; --`), apagar ou vazar dados. Os **placeholders** (`:nome`, `?` ou `%(nome)s`) fazem o driver escapar/enviar os valores separadamente do comando, então o banco os trata apenas como **dados**, nunca como SQL.

## 🟡 Nível 2 — SQLAlchemy Core (Automatização Programática)

In [ ]:
# Passo 1: tabela 'projetos' definida de forma programática
metadata = MetaData()

projetos = Table(
    'projetos', metadata,
    Column('id', Integer, primary_key=True),
    Column('nome_projeto', String(100), nullable=False),
    Column('responsavel', String(100)),
    Column('orcamento', Float)
)

metadata.create_all(engine)
print("Tabela 'projetos' criada via metadata.create_all().")

In [ ]:
# Passo 2: inserção em lote (bulk insert) a partir de lista de dicionários
lista_projetos = [
    {"nome_projeto": "Portal RH", "responsavel": "Ana Souza", "orcamento": 45000},
    {"nome_projeto": "Biometria", "responsavel": "Carlos Lima", "orcamento": 120000},
    {"nome_projeto": "Banco de Talentos", "responsavel": "Ana Souza", "orcamento": 30000},
]

with engine.begin() as conn:
    conn.execute(insert(projetos), lista_projetos)
print(f"{len(lista_projetos)} projetos inseridos em lote.")
pd.read_sql_query("SELECT * FROM projetos", engine)

In [ ]:
# Passo 3: reajuste para 'Desenvolvedor Júnior' (+R$ 500) usando update().where().values()
# Refletir a tabela criada via SQL puro para usá-la no Core (Column-style)
funcionarios = Table('funcionarios', metadata, autoload_with=engine)

with engine.begin() as conn:
    resultado = conn.execute(
        update(funcionarios)
        .where(funcionarios.c.cargo == 'Desenvolvedor Júnior')
        .values(salario=funcionarios.c.salario + 500)
    )
print(f"Linhas atualizadas: {resultado.rowcount}")
pd.read_sql_query("SELECT * FROM funcionarios", engine)

In [ ]:
# Passo 4: relatório salarial — média por cargo com func.avg() e group_by()
stmt_relatorio = (
    select(func.avg(funcionarios.c.salario).label('salario_medio'), funcionarios.c.cargo)
    .group_by(funcionarios.c.cargo)
    .order_by(func.avg(funcionarios.c.salario).desc())
)

with engine.connect() as conn:
    linhas = conn.execute(stmt_relatorio).all()

df_relatorio = pd.DataFrame(linhas, columns=['salario_medio', 'cargo'])
df_relatorio

## 🔵 Nível 3 — ORM (Orientação a Objetos e Relacionamentos)

In [ ]:
# Passo 1: classes Python mapeando tabelas
Base = declarative_base()

class Departamento(Base):
    __tablename__ = 'departamentos'

    id = mapped_column(Integer, primary_key=True)
    nome = mapped_column(String(100), nullable=False)
    funcionarios = relationship("FuncionarioORM", back_populates="departamento")

class FuncionarioORM(Base):
    __tablename__ = 'funcionarios_orm'

    id = mapped_column(Integer, primary_key=True, autoincrement=True)
    nome = mapped_column(String(100), nullable=False)
    cargo = mapped_column(String(100), nullable=False)
    salario = mapped_column(Float, nullable=False)
    departamento_id = mapped_column(Integer, ForeignKey('departamentos.id'))
    departamento = relationship("Departamento", back_populates="funcionarios")

Base.metadata.create_all(engine)
print("Classes ORM criadas e tabelas 'departamentos'/'funcionarios_orm' persistidas.")

In [ ]:
# Passo 3: fábrica de sessões + persistência em lote
SessionFabrica = sessionmaker(bind=engine)
sessao = SessionFabrica()

# Criar o departamento e já adicionar funcionários a ele (relacionamento bidirecional)
departamento_ti = Departamento(nome='TI')
departamento_ti.funcionarios = [
    FuncionarioORM(nome='Bruno Carvalho', cargo='Dev Pleno', salario=8000.00),
    FuncionarioORM(nome='Julia Rocha', cargo='Dev Pleno', salario=8200.00),
    FuncionarioORM(nome='Pedro Martins', cargo='Analista de Suporte', salario=4500.00),
]

sessao.add(departamento_ti)   # adiciona o departamento (e os funcionários em cascata na relação)
sessao.commit()                # persiste tudo de uma só vez
print("Departamento 'TI' com 3 funcionários persistidos.")

In [ ]:
# Passo 4: consulta orientada a objetos → lista de objetos FuncionarioORM
stmt_ti = (
    select(FuncionarioORM)
    .join(Departamento)
    .where(Departamento.nome == 'TI')
)

funcionarios_ti = sessao.execute(stmt_ti).scalars().all()
print(f"Funcionários do departamento TI ({len(funcionarios_ti)}):")
for f in funcionarios_ti:
    print(f"  • {f.nome} — {f.cargo} — R$ {f.salario:,.2f} | depto: {f.departamento.nome}")

sessao.close()
print("\nSessão fechada.")

## 🏁 Checklist

- [ ] N1: engine + CREATE TABLE + INSERT com placeholders + `read_sql_query`
- [ ] N2: Core (`Table`/`MetaData`), bulk insert, update com `.where().values()`, relatório `avg`/`group_by`
- [ ] N3: ORM (`declarative_base`, `relationship`/`ForeignKey`), sessão, consulta `.scalars().all()`

Bom estudo! 🎉